# Laboratorio 2. Deep Learning y catch22

CC3084 Data Science, Universidad del Valle de Guatemala, Semestre II 2026, Sección 20

Diego López, Nelson Escalante, Roberto Nájera

Ingreso de viajeros internacionales a Guatemala entre 2009 y junio de 2026. Este trabajo
continúa el Laboratorio 1 y reutiliza sus mismas series con las mismas particiones de
entrenamiento y prueba.

## Preparación

Las celdas de este documento leen los resultados que produce el pipeline con
`python main.py all` y los presentan en tablas y figuras. Ningún número se calcula aquí ni
se escribe a mano, porque todos provienen de `results/*.json`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RAIZ = Path.cwd()
RESULTS = RAIZ / "results"


def cargar(nombre, pendiente_de=None):
    """Lee un JSON de results/. Avisa en vez de reventar si aun no existe."""
    ruta = RESULTS / f"{nombre}.json"
    if not ruta.exists():
        quien = f" Lo genera {pendiente_de}." if pendiente_de else ""
        display(Markdown(f"> Todavia no existe `results/{nombre}.json`.{quien}"))
        return None
    with open(ruta, encoding="utf-8") as fh:
        return json.load(fh)


def figura(ruta_relativa, ancho=760):
    display(Image(filename=str(RAIZ / ruta_relativa), width=ancho))


def tabla(filas, columnas):
    display(pd.DataFrame(filas, columns=columnas))

---
# 1. Modelos LSTM

## 1.1 Series y particiones utilizadas

El enunciado pide trabajar dos de las series del laboratorio anterior, con los mismos
conjuntos de entrenamiento y prueba.

In [ ]:
split = cargar("split")
lstm = cargar("lstm")

if split and lstm:
    tabla(
        [[s["clave"], split["train"]["inicio"], split["train"]["fin"],
          split["train"]["n_meses"], split["test"]["inicio"], split["test"]["fin"],
          split["test"]["n_meses"]]
         for s in lstm["series"]],
        ["Serie", "Train desde", "Train hasta", "Meses train",
         "Test desde", "Test hasta", "Meses test"],
    )

Se eligieron la serie total y la vía Aérea.

La vía Marítima quedó fuera por una razón concreta. Sus últimos doce meses de entrenamiento
valen cero exacto porque las fronteras marítimas estuvieron cerradas, así que una red con
ventana de doce meses solo observa ceros al momento de pronosticar y su predicción recursiva
colapsa a cero. Probamos cinco configuraciones distintas y ninguna evitó ese
comportamiento. La serie sí participa del ejercicio 2.

## 1.2 Configuraciones y tuneo de parámetros

Dos configuraciones por serie. Una red pequeña con ventana de un año y otra más grande con
ventana de dos, para observar si el contexto adicional compensa el riesgo de sobreajuste.

In [ ]:
if lstm:
    filas = []
    for s in lstm["series"]:
        for nombre, info in s["modelos"].items():
            p = info["parametros"]
            filas.append([s["clave"], nombre, p["ventana"], p["unidades"], p["capas"],
                          p["dropout"], p["epochs_usadas"], round(p["loss_final"], 4)])
    tabla(filas, ["Serie", "Config", "Ventana", "Unidades", "Capas", "Dropout",
                  "Epocas elegidas", "Perdida final"])

### Tuneo

El número de épocas se eligió validando contra los últimos doce meses del entrenamiento.
El conjunto de prueba no interviene en la selección, porque usarlo sería fuga de datos y las
métricas finales dejarían de medir lo que dicen medir.

In [ ]:
if lstm:
    for s in lstm["series"]:
        for nombre, info in s["modelos"].items():
            t = info["parametros"]["tuneo"]
            if not t:
                continue
            display(Markdown(f"**{s['clave']} — {nombre}** "
                             f"(validacion {t['val_inicio']} a {t['val_fin']}, "
                             f"{t['n_muestras_interno']} muestras)"))
            tabla([[r["epochs"], round(r["rmse_val"], 4),
                    "elegido" if r["epochs"] == t["mejor"] else ""]
                   for r in t["resultados"]],
                  ["Epocas", "RMSE validacion", ""])

Los doce meses de validación van de abril de 2020 a marzo de 2021, es decir que caen dentro
del colapso pandémico. El número de épocas que se elige queda entonces sesgado hacia predecir
bien una caída. No hay forma de evitarlo con 147 observaciones y la pandemia ubicada al final
del tramo de entrenamiento.

## 1.3 Predicción con el mejor modelo

In [ ]:
comp = cargar("comparison")

if comp and lstm:
    claves = [s["clave"] for s in lstm["series"]]
    for s in comp["series"]:
        if s["clave"] not in claves:
            continue
        display(Markdown(f"### {s['nombre']} — gana **{s['ganador']['modelo']}**"))
        filas = sorted(([m, round(r["mae"], 1), round(r["rmse"], 1)]
                        for m, r in s["modelos"].items()), key=lambda f: f[2])
        tabla(filas, ["Modelo", "MAE", "RMSE"])
        figura(s["fig_forecast"])

## 1.4 Comparación con los modelos del laboratorio anterior

La comparación usa RMSE sobre las mismas 63 observaciones de prueba y la misma partición del
laboratorio anterior, con MAE como respaldo. El criterio de selección que quedó fijado
entonces en `comparison.json` sigue vigente, de modo que los siete modelos compiten bajo la
misma regla y no hubo que redefinir la metodología a mitad de camino.

In [ ]:
if comp and lstm:
    filas = []
    for s in comp["series"]:
        if s["clave"] not in [x["clave"] for x in lstm["series"]]:
            continue
        mods = s["modelos"]
        clasico = min((m for m in mods if not m.startswith("lstm")),
                      key=lambda m: mods[m]["rmse"])
        gan = s["ganador"]["modelo"]
        filas.append([s["clave"], gan, round(mods[gan]["rmse"], 1),
                      clasico, round(mods[clasico]["rmse"], 1),
                      f"{(1 - mods[gan]['rmse'] / mods[clasico]['rmse']) * 100:.1f}%",
                      f"{(1 - mods[gan]['mae'] / mods[clasico]['mae']) * 100:.1f}%"])
    tabla(filas, ["Serie", "Mejor LSTM", "RMSE LSTM", "Mejor clásico", "RMSE clásico",
                  "Reducción RMSE", "Reducción MAE"])

Los LSTM predicen mejor en las dos series. El mejor de ellos reduce el error a la mitad en la
serie total y en más de un tercio en la vía Aérea. Tampoco se trata solo del ganador, porque
las cuatro configuraciones quedan por encima de los cinco modelos del laboratorio anterior.

Ahora, la ventaja no viene de haber capturado el patrón estacional sino de acertar el nivel.
Los modelos clásicos extrapolan desde las últimas observaciones que vieron, y el
entrenamiento termina en marzo de 2021, en el punto más bajo de la pandemia. Proyectan
entonces la continuación de ese piso y se quedan por debajo de los 120,000 viajeros mensuales
cuando la serie real promedia 276,685. El LSTM aprendió de los datos previos que a un valor
muy bajo le siguen valores más altos, así que su pronóstico se estabiliza en un nivel
cercano al observado.

Ninguno de los siete reproduce la estacionalidad del período de prueba. Los pronósticos de
los LSTM son curvas suaves y el diagnóstico de aplanamiento marca `aplanado = True` en las
cuatro configuraciones. Con un error de nivel que supera los 150,000 viajeros, tener los
picos anuales en el lugar correcto no compensa la diferencia.

Todo esto se determinó comparando RMSE y MAE de los siete modelos sobre las mismas 63
observaciones, con los valores que ya estaban calculados en `comparison.json`. Las dos
métricas apuntan en la misma dirección en ambas series, lo cual descarta que la conclusión
dependa de cuál se elija.

---
# 2. Exploración de similitud con catch22

## 2.1 Qué es catch22 y por qué importa

catch22 resume una serie de tiempo en 22 números que describen su comportamiento. Miden
autocorrelación, la forma de la distribución de valores, la longitud de las rachas, el
contenido de frecuencias y la presencia de valores extremos, entre otras cosas. Cada serie
deja de ser una secuencia de largo variable y pasa a ser un vector de tamaño fijo.

Con siete series todavía es viable comparar los gráficos uno por uno. Con setenta deja de
serlo, y ahí está la utilidad de ese cambio de representación. Convertidas en filas de una matriz, las series
admiten las herramientas habituales de datos tabulares como componentes principales,
agrupamiento y distancias, y las preguntas sobre similitud empiezan a tener respuestas
medibles en vez de depender del criterio de quien observa.

Las 22 características salieron de un catálogo de más de siete mil candidatas que evaluaron
Fulcher y colaboradores. Se quedaron con estas por dos razones. Distinguen bien entre series
de distinto tipo y, sobre todo, no se repiten entre sí, de manera que cada una aporta información
que las demás no capturan.

## 2.2 a 2.4 Extracción de características y matriz

Se extraen las 22 características de las siete series del laboratorio anterior, tomadas sobre
el tramo de entrenamiento. Con eso se arma la matriz donde cada fila es una serie y cada
columna una característica.

In [ ]:
c22 = cargar("catch22")

if c22:
    display(Markdown(f"Matriz de **{c22['n_series']} series x {c22['n_features']} "
                     f"caracteristicas**."))
    display(pd.DataFrame(c22["matriz"], index=c22["series"],
                         columns=c22["features"]).round(3))

### Estandarización

Las 22 características viven en escalas muy distintas, en estas series van de −0.94 a 49, así
que se estandarizan por columna antes de cualquier análisis comparativo. Si se estandarizara
con una sola media para toda la matriz, las características de rango grande dominarían las
distancias y el PCA, y el resto dejaría de influir.

In [ ]:
if c22:
    display(pd.DataFrame(c22["matriz_estandarizada"], index=c22["series"],
                         columns=c22["features"]).round(3))

## 2.5 Análisis de la matriz

Sobre la matriz estandarizada se aplican las cinco técnicas que pide el enunciado:
componentes principales, agrupamiento, mapa de calor, correlaciones entre características y
distancias entre series.

In [ ]:
analisis = cargar("catch22_analysis", pendiente_de="el analisis multivariado (2.5)")

TITULOS = {
    "pca": "Análisis de componentes principales",
    "clusters": "Agrupamiento de series",
    "heatmap": "Mapa de calor de las características",
    "correlaciones": "Correlaciones entre características",
    "distancias": "Distancias entre series",
}

if analisis:
    pca = analisis.get("pca", {})
    ve = pca.get("varianza_explicada")
    if ve:
        acum = sum(ve[:2]) * 100
        display(Markdown(f"Los dos primeros componentes explican **{acum:.1f}%** de la "
                         f"variación total."))
    cl = analisis.get("clustering", {})
    if cl:
        display(Markdown(f"Agrupamiento por **{cl.get('metodo', 'n/d')}** con "
                         f"**{cl.get('n_clusters', 'n/d')}** grupos."))
        tabla(sorted(cl.get("asignacion", {}).items()), ["Serie", "Grupo"])

    for clave, ruta in analisis.get("figuras", {}).items():
        display(Markdown(f"**{TITULOS.get(clave, clave)}**"))
        figura(ruta)

El primer componente separa con claridad a Marítima y Estados Unidos del resto. La lectura
detallada de cada figura está en los incisos que siguen.

## 2.6 – 2.13 Análisis e interpretación

In [ ]:
if analisis:
    d = analisis["distancias"]
    ser, Mx = d["series"], d["matriz"]
    pares = sorted((Mx[i][j], ser[i], ser[j])
                   for i in range(len(ser)) for j in range(i + 1, len(ser)))
    tabla([[a, b, round(v, 3)] for v, a, b in pares[:4]],
          ["Serie A", "Serie B", "Distancia"])
    display(Markdown("_Los cuatro pares más cercanos._"))
    tabla([[s, round(sum(Mx[i][j] for j in range(len(ser)) if j != i) / (len(ser) - 1), 3)]
           for i, s in enumerate(ser)],
          ["Serie", "Distancia media al resto"])
    display(Markdown("_Distancia media de cada serie al resto: cuanto más alta, más atípica._"))

### 2.7 ¿Cuáles series presentan comportamientos más similares?

Terrestre y El Salvador forman el par más parecido, con la distancia euclidiana más baja de
las veintiuna combinaciones posibles, 2.784. Después vienen total con Guatemala en 3.742 y
total con Terrestre en 4.051.

El resultado no es casualidad numérica sino consecuencia de cómo está armado el dataset. El
Salvador es el principal mercado emisor y entra casi por completo por vía terrestre, de manera
que ambas series miden en gran parte el mismo flujo de personas. Algo similar ocurre entre la serie
total y Terrestre, que aporta el 61% del volumen histórico.

Las tres caen en la misma región del plano de componentes principales, todas con valores
negativos en el primer componente.

### 2.8 ¿Qué características fueron las más importantes para diferenciar las series?

El primer componente concentra el 56.1% de la variación y lo dominan características de
estructura temporal, con pesos muy parejos entre 0.266 y 0.282. Las cuatro de mayor peso son
`FC_LocalSimple_mean3_stderr`, que mide el error de predicción a un paso con media móvil,
`SB_BinaryStats_mean_longstretch1`, la racha más larga por encima de la media, `CO_f1ecac`,
el primer cruce de la autocorrelación por 1/e, y `SP_Summaries_welch_rect_centroid`, el
centroide del espectro de frecuencias.

El segundo componente aporta un 18.4% y se apoya en dos características con peso 0.43,
`PD_PeriodicityWang_th0_01` para periodicidad y `DN_HistogramMode_5` para la moda de la
distribución.

Ninguna característica individual explica la separación entre series. La produce el conjunto
de varias medidas de persistencia y estructura temporal actuando juntas, lo cual explica que los pesos
del primer componente estén tan repartidos.

### 2.9 ¿Existen grupos naturales de series?

Sí, y son dos. El agrupamiento jerárquico evaluó k igual a 2, 3 y 4, y se quedó con k = 2
porque dio el mayor coeficiente de silueta, 0.366.

El primer grupo reúne a Marítima y Estados Unidos de América. El segundo a total, Aérea,
Terrestre, El Salvador y Guatemala.

Lo que comparten las dos series del primer grupo no es el volumen, aunque sean las de menor
volumen, sino la estructura de sus interrupciones. Ambas tienen meses en cero, con una racha
de doce en Marítima y de cinco en Estados Unidos, y son también las dos con mayor distancia
media al resto. El segundo grupo reúne series de flujo continuo.

Un coeficiente de 0.366 habla de una separación moderada. Los grupos existen pero no están
del todo aislados, algo esperable con solo siete observaciones.

### 2.10 ¿Las series de una misma categoría tienden a agruparse?

No, y tiene una consecuencia práctica. Agrupar la planificación por vía de ingreso reúne bajo
un mismo criterio a Marítima y Aérea, que se comportan de forma distinta.

Las tres vías de ingreso terminan separadas, con Aérea y Terrestre en el segundo grupo y
Marítima en el primero. A los tres países les pasa lo mismo, El Salvador y Guatemala juntos y
Estados Unidos aparte.

La categoría administrativa, entonces, no predice el comportamiento temporal. Marítima se
parece más a Estados Unidos, que es una serie de país, que a las otras dos vías con las que
comparte clasificación. Lo que agrupa a las series es cómo se mueven en el tiempo y no la
etiqueta con que la fuente las organiza.

### 2.11 ¿Qué serie presenta el comportamiento más atípico?

Marítima, con una distancia media al resto de 8.947, la más alta de las siete. Aparece además
en el extremo del primer componente principal con +5.956, mientras que las cinco series del
segundo grupo están todas en valores negativos.

Estas son las características que más la separan del promedio del resto.

| Característica | Marítima | Resto |
|---|---:|---:|
| `DN_OutlierInclude_n_001_mdrmd` | −2.45 | +0.41 |
| `FC_LocalSimple_mean1_tauresrat` | +2.08 | −0.35 |
| `MD_hrv_classic_pnn40` | −2.07 | +0.34 |

La causa está en los datos. Marítima tiene doce meses consecutivos en cero por
el cierre de fronteras marítimas y perdió detalle de registro desde 2017. Un tramo plano en
cero altera al mismo tiempo la distribución de valores, la estructura de rachas y la
predictibilidad local, que es exactamente lo que miden esas tres características.

La segunda más atípica es Estados Unidos de América con 8.119, y también tiene meses en cero.

### 2.12 ¿Los agrupamientos son consistentes con el análisis exploratorio?

La tabla contrasta la pertenencia a cada grupo con las métricas que se calcularon en el
laboratorio anterior.

In [ ]:
if comp:
    filas = []
    for cat in ("vias", "paises"):
        for k, v in comp["comparativo"][cat]["detalle"].items():
            grupo = analisis["clustering"]["asignacion"].get(k) if analisis else None
            filas.append([k, "A" if grupo == 0 else "B",
                          round(v["fuerza_estacionalidad"], 3), round(v["cv"], 3),
                          f"{v['pendiente_anual']:,.0f}", v["racha_ceros_max"]])
    tabla(filas, ["Serie", "Grupo", "Fuerza estacional", "Coef. variación",
                  "Pendiente anual", "Meses en cero"])

En volatilidad e interrupciones los dos análisis coinciden. Las dos series del primer grupo
son justamente las que tienen meses en cero, y Marítima es además la de mayor coeficiente de
variación con 0.868, frente al rango de 0.330 a 0.548 del resto.

También coinciden en lo que respecta a la autocorrelación. El primer componente está dominado
por medidas de persistencia como `CO_f1ecac` y la longitud de rachas, y esas son las mismas
propiedades que describen las funciones de autocorrelación del laboratorio anterior.

En estacionalidad, en cambio, los dos análisis discrepan. Según la fuerza estacional que se
calculó antes, las más estacionales son Estados Unidos con 0.703 y Marítima con 0.614, y
ambas terminaron en el mismo grupo. Pero Aérea con 0.558 y Guatemala con 0.534 tienen valores
intermedios y quedaron en el otro. La estacionalidad no es lo que separa los grupos.

Con la tendencia pasa algo similar. Terrestre tiene la mayor pendiente, 12,373 viajeros al
año, y Marítima la menor con 286, pero ninguna de las dos variables determina la pertenencia
a un grupo. Estados Unidos, cuya pendiente es de 541, está junto a Marítima, mientras
Guatemala con 6,946 está en el grupo opuesto.

Marítima fue la única vía que cerró por completo, y esa interrupción es precisamente lo que la
vuelve atípica en catch22.

### 2.13 Tres descubrimientos que el análisis exploratorio tradicional no mostraba

El primero es que la categoría administrativa no predice el comportamiento. El exploratorio
del laboratorio anterior analizó vías y países como grupos separados, dando por sentado que
cada categoría es homogénea. catch22 muestra que no lo es, porque Marítima se parece más a
Estados Unidos que a las otras dos vías. Para el INGUAT eso implica que segmentar la
planificación por vía o por país puede juntar series que se comportan distinto.

El segundo es que Terrestre y El Salvador son casi la misma serie. Su distancia de 2.784 es la
menor de las veintiuna. El exploratorio las trató como análisis independientes, uno por vía y
otro por país, sin cuantificar cuánto se solapan. Modelarlas por separado resulta en buena
medida redundante.

El tercero es que lo que hace atípica a Marítima es la estructura de sus interrupciones y no
su volumen. El exploratorio la había identificado como la vía de menor peso, con 2.4% del
total. catch22 muestra que su rareza no viene del tamaño sino de las rachas en cero, porque
las tres características que más la separan miden valores extremos, predictibilidad local y
variabilidad entre observaciones consecutivas, y ninguna tiene que ver con el nivel de la
serie.

Queda una reserva sobre la matriz de correlaciones. Con siete series y veintidós
características la matriz de datos tiene rango 6 y la de correlaciones sale
singular y muchos coeficientes valen ±1 por construcción y no por relación real. Los
agrupamientos y las distancias no se ven afectados, porque se calculan sobre las series y no
sobre las características, pero las correlaciones individuales entre características no
admiten interpretación con esta cantidad de observaciones.

## 2.14 Modelo LSTM con las características de catch22

Se entrena una red que, además de la ventana de doce meses, recibe el vector de 22
características concatenado al estado oculto justo antes de la capa de salida. La comparación
es contra el mejor de los dos LSTM ya ajustados, sobre el mismo conjunto de prueba.

In [ ]:
modelo_c22 = cargar("catch22_modelo", pendiente_de="el modelo del inciso 2.14")

if modelo_c22:
    filas = []
    for s in modelo_c22["series"]:
        c22m, prev = s["lstm_catch22"], s["mejor_lstm_existente"]
        filas.append([s["clave"], prev["modelo"], round(prev["rmse"], 1), round(prev["mae"], 1),
                      round(c22m["rmse"], 1), round(c22m["mae"], 1),
                      f"{(c22m['rmse'] / prev['rmse'] - 1) * 100:+.1f}%",
                      "sí" if s["gana_lstm_catch22"] else "no"])
    tabla(filas, ["Serie", "Mejor LSTM previo", "RMSE previo", "MAE previo",
                  "RMSE catch22", "MAE catch22", "Cambio RMSE", "¿Gana catch22?"])

In [ ]:
if modelo_c22:
    for s in modelo_c22["series"]:
        p = s["lstm_catch22"]["parametros"]
        lb = s["lstm_catch22"]["ljung_box"]
        display(Markdown(
            f"**{s['clave']}** — épocas {p['epochs_usadas']}, "
            f"pérdida final {p['loss_final']:.4f}, "
            f"{p['n_catch22_features']} características agregadas. "
            f"Ljung-Box p={lb['pvalue']:.3g} "
            f"(residuos {'independientes' if lb['independientes'] else 'con estructura'})."))

Las características de catch22 no mejoran la predicción. El error sube en las dos series, un
1.6% en total y un 33% en Aérea.

Las 22 características describen la serie completa, no el mes que se quiere predecir. Su valor
es idéntico en cada paso del pronóstico, así que la red recibe veintidós constantes que no le
sirven para distinguir un mes del siguiente y, a cambio, suma parámetros que puede
sobreajustar con las 135 muestras disponibles.

catch22 sirve para comparar series entre sí, que es lo que se hizo en los incisos anteriores.
No para predecir dentro de una misma serie.